# OCVWorkChain: r2SCAN//PBEsol

For QE<=7.5, where meta-GGA stress is bugged.

Two-stage workflow: 

**stage 1** relaxes the discharged and charged structures with plain PBEsol using a norm-conserving pseudopotential family (meta-GGA in Quantum
ESPRESSO requires norm-conserving pseudopotentials)

**stage 2** runs single r2SCAN SCF calculations on the relaxed structures and calculates the average voltage. 
The `ocv_parameters['for_r2scan']` flag makes stage 2 bypass the workchain's calculation-reuse cache so the SCFs genuinely run with the r2SCAN functional. 
For running the *entire* workflow (relaxations included) at r2SCAN in a single submission, see `Submit_r2SCAN_full.ipynb`.

Requires aiida-open_circuit_voltage >= 0.7 with aiida-quantumespresso 5.x

Requirements: Quantum ESPRESSO compiled with LibXC (r2SCAN is addressed through the LibXC `input_dft` string), and a norm-conserving pseudopotential family installed (e.g. `PseudoDojo/0.4/PBEsol/SR/stringent/upf` via `aiida-pseudo install pseudo-dojo`). 
The example structures used below are bundled with the repository as an AiiDA archive and can be imported with

```
verdi archive import test/structures.aiida
```

or load your own structure instead, e.g. `orm.StructureData(ase=ase.io.read('my_cathode.cif'))`.

## Loading libraries

In [ ]:
from aiida import load_profile, orm
## Indicate your profile name here
your_profile_name = 'develop'
load_profile(your_profile_name)
from aiida.plugins import WorkflowFactory
from aiida.engine import submit

## Code, constants and structure

In [ ]:
## Code and constants
code = orm.load_code(label='pw_qe-7.5@alps')

## default resources
time, num_machines, num_mpiprocs_per_machine, num_cores_per_mpiproc, npool = 83200, 2, 128, 1, 8

R2SCAN_PSEUDO_FAMILY = 'PseudoDojo/0.4/PBEsol/SR/stringent/upf'   # norm-conserving (meta-GGA requirement)
R2SCAN_FUNCTIONAL = 'XC-000I-000I-000I-000I-497L-498L'            # r2SCAN via LibXC

## Load a bundled test structure (import test/structures.aiida first) or your own
structure = orm.load_node('096d9d96-7f66-442b-a27f-2660572808ea') # LiCoO2
bulk_cation_structure = orm.load_node('faea3c48-789f-4076-af73-0cf9242bb7b2') # Li

## Stage 1: PBEsol preparation

In [ ]:
## STAGE 1: PBEsol relaxations with the r2SCAN (norm-conserving) pseudos
## Average-only, stringent protocol. 
## This is a plain-GGA OCVWorkChain whose only purpose is to produce relaxed structures and 
## PBEsol bulk-cation energy with the same pseudopotentials that the r2SCAN SCFs will use.

OCVWorkChain = WorkflowFactory('quantumespresso.ocv.ocvwc')

overrides = {
    'ocv_parameters': {'distance': 8.0, 'do_low_SOC_OCV': False, 'do_high_SOC_OCV': False},
    'ocv_relax': {'base_relax': {'pseudo_family': R2SCAN_PSEUDO_FAMILY}},   # aiida-qe 5: single PwRelax namespace
    'scf':       {'pseudo_family': R2SCAN_PSEUDO_FAMILY},
}

builder = OCVWorkChain.get_builder_from_protocol(
    code=code, structure=structure, bulk_cation_structure=bulk_cation_structure,
    protocol='stringent', overrides=overrides)
builder.clean_workdir = orm.Bool(False)

pw_dict = builder.ocv_relax.base_relax.pw
pw_dict.metadata['options']['max_wallclock_seconds'] = time
pw_dict.metadata['options']['resources']['num_machines'] = num_machines
pw_dict.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
pw_dict.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
pw_dict.parameters['ELECTRONS']['electron_maxstep'] = 100
pw_dict.parallelization = orm.Dict(dict={'npool': npool})

builder.scf.pw.metadata['options']['max_wallclock_seconds'] = 1800
builder.scf.pw.metadata['options']['resources']['num_machines'] = 1
builder.scf.pw.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
builder.scf.pw.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
builder.scf.pw.parallelization = orm.Dict(dict={'npool': max(1, npool // num_machines)})

prep_node = submit(builder)
print(f'Submitted stage-1 (PBEsol prep) OCVWorkChain PK={prep_node.pk}')

## Stage 2: r2SCAN single-point energies

In [ ]:
## Helpers for stage 2 (run after stage 1 has finished)
from aiida.common.links import LinkType
import copy

OCVWorkChain = WorkflowFactory('quantumespresso.ocv.ocvwc')

def pbesol_relaxed_unitcells(workchain):
    """Return (discharged, charged) relaxed structures from a finished stage-1 WorkChain."""
    wc = workchain if isinstance(workchain, orm.WorkChainNode) else orm.load_node(workchain)
    assert wc.is_finished_ok, f'stage-1 OCVWorkChain<{wc.pk}> has not finished successfully'
    called = {link.link_label: link.node
              for link in wc.base.links.get_outgoing(link_type=LinkType.CALL_WORK).all()}
    return (called['discharged_relax'].outputs.output_structure,
            called['charged_relax'].outputs.output_structure)

def r2scan_bulk_cation_energy_per_atom(r2scan_wc):
    """Return per-atom total energy (eV) from a finished bulk-cation r2SCAN SCF WokrChain."""
    wc = r2scan_wc if isinstance(r2scan_wc, orm.WorkChainNode) else orm.load_node(r2scan_wc)
    scf = {l.link_label: l.node
           for l in wc.base.links.get_outgoing(link_type=LinkType.CALL_WORK).all()}['bulk_cation_scf']
    d = scf.outputs.output_parameters.get_dict()
    return (d['energy'] - d.get('energy_smearing', 0.0)) / len(scf.inputs.pw.structure.sites)

In [ ]:
## STAGE 2: r2SCAN SCF on the PBEsol-relaxed structures
## `for_r2scan=True` bypasses the UUID-based reuse of the pre-relaxed branches and of the bulk
## cation, so every SCF genuinely runs with the r2SCAN functional given in the inputs.
STAGE1_PK = prep_node.pk

discharged, charged = pbesol_relaxed_unitcells(STAGE1_PK)

r2scan_pw = {'pseudo_family': R2SCAN_PSEUDO_FAMILY,
             'pw': {'parameters': {'SYSTEM':    {'input_dft': R2SCAN_FUNCTIONAL},
                                   'ELECTRONS': {'mixing_beta': 0.3}}}}
## For spin-polarised runs add 'CONTROL': {'tstress': False} above, note that meta-GGA 
## stress is not implemented for nspin=2 in Quantum ESPRESSO and `pw.x` aborts otherwise.

overrides = {
    'ocv_parameters': {'cation': 'Li', 'do_low_SOC_OCV': False, 'do_high_SOC_OCV': False,
                       'for_r2scan': True, 'volume_change_stability': False},
    'ocv_relax': {'base_relax': copy.deepcopy(r2scan_pw)},
}
## Suggestion: after the first run, replace the bulk SCF with the stored energy:
# overrides['ocv_parameters']['DFT_energy_bulk_Li'] = r2scan_bulk_cation_energy_per_atom(<r2scan PK>)
# and drop bulk_cation_structure from the builder call below.

stage1 = orm.load_node(STAGE1_PK)
builder = OCVWorkChain.get_builder_from_protocol(
    code=code,
    structure=stage1.inputs.structure,            # original discharged unitcell
    bulk_cation_structure=bulk_cation_structure,  
    discharged_unitcell_relaxed=discharged,
    charged_unitcell_relaxed=charged,
    protocol='balanced', overrides=overrides)
builder.clean_workdir = orm.Bool(False)

pw_dict = builder.ocv_relax.base_relax.pw
pw_dict.metadata['options']['max_wallclock_seconds'] = time
pw_dict.metadata['options']['resources']['num_machines'] = num_machines
pw_dict.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
pw_dict.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
pw_dict.parameters['ELECTRONS']['electron_maxstep'] = 200
pw_dict.parallelization = orm.Dict(dict={'npool': npool})

builder.scf.pw.metadata['options']['max_wallclock_seconds'] = 3600
builder.scf.pw.metadata['options']['resources']['num_machines'] = 1
builder.scf.pw.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
builder.scf.pw.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
builder.scf.pw.parallelization = orm.Dict(dict={'npool': max(1, npool // num_machines)})

r2scan_node = submit(builder)
print(f'Submitted stage-2 (r2SCAN SCF) OCVWorkChain PK={r2scan_node.pk}')

## Results

In [ ]:
node = orm.load_node(PK)   # the stage-2 OCVWorkChain
print(node.outputs.open_circuit_voltages.get_dict())   # r2SCAN//PBEsol average OCV